# English-German Translation using FLAN-T5 and WMT16 Dataset

## Introduction
In this notebook, we will use the FLAN-T5 model to perform translation from English to German using the WMT16 dataset. We will preprocess the data, fine-tune the model, and evaluate its performance using the BLEU metric.

In [ ]:
# Environment setup for a text-only FLAN-T5 notebook.
# torchvision is not required here. A broken torchvision/PyTorch pairing can
# cause: RuntimeError: operator torchvision::nms does not exist
#
# Run this cell BEFORE importing torch or transformers.
%pip uninstall -y torchvision
%pip install -q -U "transformers>=4.41,<5" "datasets>=2.19,<4" accelerate sentencepiece sacrebleu

print("Setup complete. If torch/transformers/torchvision had already been imported in this runtime, restart the runtime once, then continue from the next cell.")


Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 153.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 7.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
timm 1.0.28 requires torchvision, which is not installed.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
Setup complete. If torch/trans

### Runtime note
This notebook intentionally does **not** install `torchvision`, because FLAN-T5 is a text model. If the setup cell reports that packages were already imported, restart the Colab runtime once before running the remaining cells.


## Choose and Preprocess Dataset

In [ ]:
from datasets import load_dataset

# Load the WMT16 English-German dataset.
# The current Hub repository is namespaced as "wmt/wmt16".
dataset = load_dataset("wmt/wmt16", "de-en")
print(dataset["train"][0])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

de-en/train-00000-of-00003.parquet:   0%|          | 0.00/282M [00:00<?, ?B/s]

de-en/train-00001-of-00003.parquet:   0%|          | 0.00/267M [00:00<?, ?B/s]

de-en/train-00002-of-00003.parquet:   0%|          | 0.00/277M [00:00<?, ?B/s]

...

de-en/test-00000-of-00001.parquet:   0%|          | 0.00/475k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4548885 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2169 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2999 [00:00<?, ? examples/s]

{'translation': {'de': 'Wiederaufnahme der Sitzungsperiode', 'en': 'Resumption of the session'}}


In [ ]:
import importlib.util
import torch
import transformers
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# torchvision should be absent in this text-only notebook.
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("torchvision installed:", importlib.util.find_spec("torchvision") is not None)

MODEL_NAME = "google/flan-t5-base"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print("Device:", DEVICE)
print("Parameters:", f"{model.num_parameters():,}")


PyTorch: 2.11.0+cu128
Transformers: 4.57.6
torchvision installed: False


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device: cuda
Parameters: 247,577,856


In [ ]:
# Tokenize source and target text using the current Transformers API.
def preprocess_data(examples): #Defines a function that preprocesses a batch of dataset examples.
    inputs = [
        f"translate English to German: {item['en']}" #Extracts every English sentence and adds a translation instruction before it.
        for item in examples["translation"]
    ] #This follows the same instruction-based preprocessing pattern used in Code 3, so it is a Repeat, except the target language is now German.
    targets = [item["de"] for item in examples["translation"]]

    return tokenizer(
        inputs,
        text_target=targets,
        max_length=128,
        truncation=True,
    )

train_size = min(30_000, len(dataset["train"])) #Selects up to 30,000 training examples
test_size = min(2_000, len(dataset["test"])) #Selects up to 2,000 test examples. Using a smaller test subset makes evaluation and text generation faster.

train_dataset = dataset["train"].select(range(train_size)).map( #Selects the first train_size examples and applies preprocessing.
    preprocess_data, #Passes the preprocessing function to .map().
    batched=True, #Sends several examples to preprocess_data() at the same time.
    remove_columns=dataset["train"].column_names, #Removes the original dataset columns after tokenization.
    desc="Tokenizing training data", #Adds a description to the progress bar displayed during tokenization.
)

test_dataset = dataset["test"].select(range(test_size)).map( #Selects up to 2,000 test examples and preprocesses them.
    preprocess_data,
    batched=True,
    remove_columns=dataset["test"].column_names, #Uses batch preprocessing and removes the original columns.
    desc="Tokenizing test data",
)

print(train_dataset)
print(train_dataset[0].keys())


Tokenizing training data:   0%|          | 0/30000 [00:00<?, ? examples/s]

Tokenizing test data:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 30000
})
dict_keys(['input_ids', 'attention_mask', 'labels'])


In [ ]:
# PyTorch models do not have model.summary().
print(model)
print("Total parameters:", f"{model.num_parameters():,}")


T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [ ]:
# Optional: freeze only the encoder.
# Freezing shared embeddings, encoder, and decoder together would leave
# almost nothing useful to fine-tune.
model.encoder.requires_grad_(False)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad) #Counts only the parameters that can be updated during training.
#model.parameters() returns the model parameters. p.requires_grad checks whether a parameter is trainable. p.numel() returns the number of values in that parameter.
total = sum(p.numel() for p in model.parameters())
#Counts all parameters, including frozen and trainable ones.
print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")


Trainable parameters: 137,949,312 / 247,577,856 (55.72%)


In [ ]:
# Show the parameters that remain trainable after freezing the encoder.
for name, parameter in model.named_parameters(): #Loops through the model’s parameters together with their names.
    if parameter.requires_grad: #This helps confirm whether the intended parts of the model remain unfrozen.
        print(name)


decoder.block.0.layer.0.SelfAttention.q.weight
decoder.block.0.layer.0.SelfAttention.k.weight
decoder.block.0.layer.0.SelfAttention.v.weight
decoder.block.0.layer.0.SelfAttention.o.weight
decoder.block.0.layer.0.SelfAttention.relative_attention_bias.weight
decoder.block.0.layer.0.layer_norm.weight
decoder.block.0.layer.1.EncDecAttention.q.weight
decoder.block.0.layer.1.EncDecAttention.k.weight
decoder.block.0.layer.1.EncDecAttention.v.weight
decoder.block.0.layer.1.EncDecAttention.o.weight
decoder.block.0.layer.1.layer_norm.weight
decoder.block.0.layer.2.DenseReluDense.wi_0.weight
decoder.block.0.layer.2.DenseReluDense.wi_1.weight
decoder.block.0.layer.2.DenseReluDense.wo.weight
decoder.block.0.layer.2.layer_norm.weight
decoder.block.1.layer.0.SelfAttention.q.weight
decoder.block.1.layer.0.SelfAttention.k.weight
decoder.block.1.layer.0.SelfAttention.v.weight
decoder.block.1.layer.0.SelfAttention.o.weight
decoder.block.1.layer.0.layer_norm.weight
decoder.block.1.layer.1.EncDecAttention.

## Train the Model

In [ ]:
from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
) #Imports the same sequence-to-sequence training components used in Code 3.

# Dynamic padding and correctly shifted decoder inputs.
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    return_tensors="pt",
)
#Creates a batch-preparation object for sequence-to-sequence training.
#padding=True: Dynamically pads each batch to the longest sequence in that batch.
#return_tensors="pt": Returns PyTorch tensors.
#model=model: Allows the collator to prepare decoder inputs correctly.
#The collator also replaces padded target positions with -100, allowing the loss function to ignore padding.

training_args = Seq2SeqTrainingArguments( #Begins defining the trainer configuration.
    output_dir="./flan_t5_en_de", #Stores model checkpoints and training outputs in this directory.
    num_train_epochs=3, #Trains for three complete passes through the training dataset.
    learning_rate=5e-5, #Sets the learning rate to 0.00005.
    per_device_train_batch_size=8, #Uses eight examples per device for training and evaluation.
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,#Accumulates gradients across two training batches before updating the model. Therefore, the effective batch size on one device is: 8*2=16. Effective batch size=8*2*Num_of_Devices
    eval_strategy="epoch", #Evaluates the model after every epoch.
    save_strategy="epoch", #save_strategy="epoch",
    save_total_limit=2, #Keeps at most two saved checkpoints.
    logging_steps=100, #Prints or records training information every 100 steps.
    predict_with_generate=True, #Instructs Seq2SeqTrainer to generate translations during prediction instead of returning only raw model logits. This is essential for decoding translations and calculating BLEU.
    generation_max_length=128, #Limits generated translations to 128 tokens during trainer-based evaluation or prediction.
    generation_num_beams=4,  #Uses beam search with four candidate generation paths.
    fp16=torch.cuda.is_available(), #Uses 16-bit mixed-precision training when CUDA is available.
    report_to="none", #Prevents external logging services such as Weights & Biases from starting automatically.
    load_best_model_at_end=True, #Restores the best saved checkpoint when training ends.
    metric_for_best_model="eval_loss", #Uses evaluation loss to identify the best checkpoint. Lower loss is considered better.
    greater_is_better=False,
)

trainer = Seq2SeqTrainer( #Creates the object that manages training and evaluation.
    model=model,
    args=training_args, #Provides the model and training configuration.
    train_dataset=train_dataset,
    eval_dataset=test_dataset, #Provides the processed training and evaluation datasets.
    data_collator=data_collator, #Uses the sequence-to-sequence collator to construct batches.
    processing_class=tokenizer, #Provides the tokenizer for processing and saving.
)

trainer.train()


Epoch,Training Loss,Validation Loss
1,0.000000,nan
2,0.000000,nan
3,0.000000,nan


TrainOutput(global_step=5625, training_loss=0.0, metrics={'train_runtime': 1477.8085, 'train_samples_per_second': 60.901, 'train_steps_per_second': 3.806, 'total_flos': 8469678097760256.0, 'train_loss': 0.0, 'epoch': 3.0})

## Evaluate the Model

In [ ]:
import numpy as np
import sacrebleu #sacrebleu for calculating the translation BLEU score.

# Make sure generation is used instead of raw logits
trainer.args.predict_with_generate = True
trainer.args.generation_max_length = 128
trainer.args.generation_num_beams = 4

#Reapplies generation settings already supplied through Seq2SeqTrainingArguments. These assignments are mostly defensive: they ensure that generation remains enabled before calling predict().

prediction_output = trainer.predict(
    test_dataset,
    max_length=128,
    num_beams=4
)
#Runs prediction on the test dataset. Unlike ordinary classification prediction, the model generates a complete German translation for each English input.
predictions = prediction_output.predictions
labels = prediction_output.label_ids
#predictions: Generated translation token IDs.
#labels: Correct German translation token IDs.
# Some Transformers versions return predictions inside a tuple
if isinstance(predictions, tuple):
    predictions = predictions[0]

#Some Transformers versions may return predictions inside a tuple. If that happens, this extracts the actual generated token sequences.

predictions = np.asarray(predictions)
labels = np.asarray(labels)
#Converts both objects into NumPy arrays so they can be cleaned and decoded.
print("Prediction shape:", predictions.shape)
print("Prediction dtype:", predictions.dtype)

# Generated predictions should be a 2-D array:
# (number_of_examples, generated_sequence_length)
if predictions.ndim != 2:
    raise ValueError(
        f"Expected generated token IDs with 2 dimensions, "
        f"but received shape {predictions.shape}. "
        "Recreate Seq2SeqTrainer with predict_with_generate=True."
    )
#Stops execution with a clear explanation if the trainer returned logits instead of generated token IDs. Raw logits would generally be three-dimensional:Generated token IDs should be two-dimensional.
# Replace ignored and invalid positions before decoding
predictions = np.where(
    predictions >= 0,
    predictions,
    tokenizer.pad_token_id
).astype(np.int64)

#Replaces negative prediction values with the padding token ID and converts the result to integer token IDs. Generated predictions normally should not contain negative values, but this is a safety check.

labels = np.where(
    labels != -100,
    labels,
    tokenizer.pad_token_id
).astype(np.int64)

#Replaces -100 values in the labels with the padding token ID. -100 tells the training loss to ignore padded positions, but the tokenizer cannot decode -100. Therefore, it must be replaced before decoding.
#This is a Repeat of the label-cleaning step from Code 3, now applied to an entire NumPy array.

#Keep token IDs within the tokenizer vocabulary
predictions = np.where(
    predictions < len(tokenizer),
    predictions,
    tokenizer.unk_token_id
)
#Replaces token IDs outside the tokenizer’s vocabulary with the unknown-token ID.
labels = np.where(
    labels < len(tokenizer),
    labels,
    tokenizer.unk_token_id
)#However, the earlier code has already removed negative values.
#Performs the same safety check for the reference labels. A more complete validity condition would check both lower and upper limits:
predicted_texts = tokenizer.batch_decode(
    predictions,
    skip_special_tokens=True
)#Converts all generated token sequences into readable German translations.
#However, the earlier code has already removed negative values.
reference_texts = tokenizer.batch_decode(
    labels,
    skip_special_tokens=True
)
#Converts the correct German label sequences into readable reference translations.
# Remove unnecessary surrounding whitespace
predicted_texts = [text.strip() for text in predicted_texts]
reference_texts = [text.strip() for text in reference_texts]
#Removes unnecessary spaces and line breaks from the beginning and end of each translation.
bleu = sacrebleu.corpus_bleu(
    predicted_texts,
    [reference_texts]
)
#Calculates corpus-level BLEU by comparing all predicted translations with their reference translations.
#The reference list is placed inside another list because SacreBLEU supports multiple valid reference translations for each prediction:
print(f"BLEU score on test subset: {bleu.score:.2f}")

for i in range(min(5, len(predicted_texts))):
    source = tokenizer.decode( #Decodes the tokenized English input back into readable text.
        test_dataset[i]["input_ids"],
        skip_special_tokens=True
    )

    print(f"\nInput: {source}")
    print(f"Reference: {reference_texts[i]}")
    print(f"Prediction: {predicted_texts[i]}")

Prediction shape: (2000, 128)
Prediction dtype: int64
BLEU score on test subset: 14.86

Input: translate English to German: Obama receives Netanyahu
Reference: Obama empfängt Netanyahu
Prediction: Obama erhält Netanyahu

Input: translate English to German: The relationship between Obama and Netanyahu is not exactly friendly.
Reference: Das Verhältnis zwischen Obama und Netanyahu ist nicht gerade freundschaftlich.
Prediction: Die Beziehung zwischen Obama und Netanyahu ist nicht ganz freundlich.

Input: translate English to German: The two wanted to talk about the implementation of the international agreement and about Teheran's destabilising activities in the Middle East.
Reference: Die beiden wollten über die Umsetzung der internationalen Vereinbarung sowie über Teherans destabilisierende Maßnahmen im Nahen Osten sprechen.
Prediction: Die beiden wollten über die Umsetzung der internationalen Vereinbarung und über die Aktivitäten der Teheran in den Nahen Osten sprechen.

Input: translat

## Conclusion
In this notebook, we used the FLAN-T5 model to perform translation from English to German using the WMT16 dataset. We preprocessed the dataset, fine-tuned the model, and evaluated its performance using the BLEU metric. The results demonstrate the effectiveness of the FLAN-T5 model for translation tasks.